In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import networkx as nx

import re
import pickle as pkl

In [ ]:
# get directory list 
myFiles = []
def absoluteFilePaths(directory):
    for dirpath,_,filenames in os.walk(directory):
        for f in filenames:
            if bool(re.search('.*graph_\d+', f)):
                myFiles.append(os.path.abspath(os.path.join(dirpath, f)))
absoluteFilePaths("/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/")
myFiles = sorted(myFiles)
for file in myFiles:
    print(file)

## Outline
    1 Construct the graph- use only actions and objects from pool for now 
    2 detect the TTPs 
    3 try to link detected TTPs together (bounded paths)
    4 Perform forward and backward exploration to get the context 
    
    Following two shells have been made useless by building the graphs offline 

In [ ]:
G = nx.MultiDiGraph()
processTree = nx.DiGraph()
#pid2actorId = {}

# subject, event, object ttriplets 
# subject - 
actionPool = ["CREATE", "MESSAGE", "ADD", "EDIT", "REMOVE", "REMOTE_CREATE", "LOAD", "OPEN"]
objectPool = ["PROCESS", "FLOW", "REGISTRY", "THREAD", "IMAGE", "MODULE"]
fileName = "/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/AIA-051-075/AIA-51-75.ecar-last.json"

# first pass scan the nodes, and add them
# Create the process treee
with open(fileName) as logFile:
    lCount = 0
    for line in logFile:
        line = line.strip()
        try:
            y = json.loads(line)
        except:
            print(line)
        #right now focus on host 051
        if y["hostname"] != "SysClient0051.systemia.com":
            continue
        
        # get the object and action
        if y["action"] not in actionPool or y["object"] not in objectPool:
            continue
        
        # now get the subject and objects 
        actorId = y["actorID"]
        pid = y["pid"]
        ppid = y["ppid"]
        objectId = y["objectID"]
        objectType = y["object"]
        G.add_node(actorId)
        G.add_node(objectId)
        G.nodes[actorId]["pid"] = pid
        G.nodes[actorId]["oType"] = "PROCESS"
        G.nodes[objectId]["oType"] = objectType
        
        processTree.add_edge(ppid, pid)

In [ ]:
G.remove_edges_from(list(G.edges))
with open(fileName) as logFile:
    lCount = 0
    for line in logFile:
        line = line.strip()
        try:
            y = json.loads(line)
        except:
            print(line)
        #right now focus on host 051
        if y["hostname"] != "SysClient0051.systemia.com":
            continue
        
        # get the object and action
        if y["action"] not in actionPool or y["object"] not in objectPool:
            continue
        
        # now get the triplet ready, subject, action, object
        properties = y["properties"]
        properties["action"] = y["action"]
        properties["principal"] = y["principal"]
        properties["timestamp"] = y["timestamp"]
        #[key = value for key,value in enum(properties)]
        G.add_edge(y["actorID"], y["objectID"], attributes = properties)
#        G.edges[y["actorID"]], properties)
#        print(properties)
        

In [ ]:
# test edges
G = nx.MultiDiGraph()
G = nx.read_gpickle(myFiles[0])
print(len(G.nodes))
print(len(G.edges))
for node in G.nodes:
    print(G.nodes[node]["oType"])
  #  print(G[node])
    break
#print(G.adj["3fb3043-fa23-44f9-9ecd-03185550fb63")
for edge in G.edges:
    print(edge[0], edge[1], edge[2])
    print(G.edges[edge]['attributes'])
    break

In [ ]:
import re
def isLanguage(regexes, text):
    for regex in regexes:
        if bool(re.search(regex, text)):
            return True
    return False


## System Discovery Techniques 

In [ ]:
# T1033 - system owner/user discovery
#    edge1- PROCESS1--OPEN->PROCESS2, where (process_name="whoami.exe" OR process_command_line="*whoami*" OR
#           process_command_line="wmic* useraccount get /ALL" OR process_name="qwinsta.exe" OR process_name="quser.exe")
#    edge2- PROCES2 --MESSAGE-->FLOW, where dst port = 389 or 636 or 445 or 8080
def huntT1033():
    T1033_e1 = []
    T1033_e2 = []
    T1033_proc = set()
    e1_proc_name = ["whoami.exe", "qwinsta.exe", "quser.exe"]
    e1_cmd_line = ["whoami", "wmic"]
    dst_ports = ['389', '636', '445', '8080']
    for edge in G.edges:
        u = edge[0]
        v = edge[1]
        attributes = G.edges[edge]['attributes']
        #print(attributes)
        if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
            isCommand = False
            try:
                command_line = attributes['command_line']
                for cl in e1_cmd_line:
                    if cl in command_line:
                        isCommand = True
                        break
            except:
                isCommand = False

            isImage = False
            try:
                image_path = attributes['image_path']
                for cl in e1_proc_name:
                    if cl in image_path:
                        isImage = True
                        break
            except:
                isImage = False

            if isCommand or isImage:
                T1033_e1.append((u, v, attributes))

    print(len(T1033_e1))

    for item in T1033_e1:
        for dst, attrs in G[item[0]].items():
            v = dst
            for key, attr in attrs.items():
                if G.nodes[v]["oType"] == "FLOW" and attr['attributes']['action'] == "MESSAGE":
                    dest_port = attr['attributes']['dest_port']
                    if dest_port in dst_ports:
                        T1033_proc.add(item[0])
                        T1033_e2.append((item[0], v, attr))

    print(len(T1033_e2))
    print(len(T1033_proc))

In [ ]:
#T1016 - Systems Network configurations discovery
# Sysmon3- Network MESSAGE	"
# (process_name="net.exe" AND, process_command_line="*net* config*") OR 
# (process_name=""ipconfig.exe"" OR ""netsh.exe"" OR ""arp.exe"" OR ""nbtstat.exe"")"
cmd_dict = ["net ", "ipconfig", "arp ", "netsh ", "nbtstat ", "ping "]
T1016_e2 = []
T1016_e1 = []
T1016_proc = set()
for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    #print(attributes)
    if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
        if 'command_line' in attributes:
            for cmd in cmd_dict:
                if cmd in attributes['command_line']:
                    #print(attributes['command_line'])
                    T1016_e1.append((u, v, attributes))
print(len(T1016_e1))
for item in T1016_e1:
    for dst, attrs in G[item[0]].items():
        v = dst
        for key, attr in attrs.items():
            if G.nodes[v]["oType"] == "FLOW" and attr['attributes']['action'] == "MESSAGE":
                T1016_proc.add(item[0])
                T1016_e2.append((item[0], v, attr))

print(len(T1016_e2))
print(len(T1016_proc))

In [ ]:
#ile = open("T1016", "w")
#or u in T1016:
#   file.write("{}\n".format(u))
#   for item in scanners:
#       if item[0] == u:
#           file.write("{}: {}\n".format(item[1], item[2]))
#   for item in messangers:
#       if item[0] == u:
#           file.write("{}: {}\n".format(item[1], item[2]))
#ile.close()

In [ ]:
# T1087 Account Discovery
# PROCESS --START-> PROCESS 
# - (image_dir="net.exe" OR "powershell.exe") AND 
# - (process_command_line="*net* user*" OR "*net* group*" OR "*net* localgroup*" OR "cmdkey*\/list*" OR 
#   "*get-localuser*" OR "*get-localgroupmembers*" OR "*get-aduser*" OR "query*user*")
T1087_proc = set()
T1087_e1 = []
image_dict = ["net.exe", "powershell.exe"]
cmd_dict = ["net.* user.*", "net.* group.*", "net.* localgroup.*", "cmdkey.*\/list.*", "get-localuser", "get-localgroupmembers", "get-aduser", "query.*user"]

for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    #print(attributes)
    if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
        isImage = False
        try:
            path = attributes['image_path']
            for img in cmd_dict:
                if img in path:
                    isImage = True
                    #print(attributes['command_line'])
        except:
            pass
            #print(attributes)

        isCmd = False
        try:
            command = attributes['command_line']
        except:
            pass
            #print(attributes)
        for cmd in cmd_dict:
            if isLanguage(cmd_dict, command):
                isCmd = True
                #print(attributes['command_line'])
                #T1087.append((u, v, attributes))

        if isCmd or isImage:
            T1087_proc.add(u)
            T1087_e1.append((u, v, attributes))

In [ ]:
print(len(T1087_e1))

In [ ]:
# T1083 File and Directory Discovery
# Sysmon event 1 PROCESS OPEN
T1083_proc = set()
T1083_e1 = []
for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    #print(attributes)
    if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
        try:
            command_line = attributes['command_line']
            if "dir " in command_line or "tree " in command_line:
                #print(attributes)
                T1083_proc.add(u)
                T1083_e1.append((u, v, attributes))
                #print(attributes['command_line'])
        except:
            pass

In [ ]:
len(T1083_e1)

In [ ]:
# T1135- Network Share Discovery
# event1 - OPEN PROCESS 
# event2 - MESSSAGE FLOW
# where image_path contains Net.exe AND
# command_line contains "*net* view*" OR "*net* share*" OR "*get-smbshare -Name*"
T1135_proc = set()
T1135_e1 = []
commands = ["net ", "*net* view*", "*net* share*", "*get-smbshare -Name*"]
for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    #print(attributes)
    if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
        isImage = False
        try:
            image_path = attributes['image_path']
            if "net.exe" in image_path:
                isImage = True
        except:
            pass
        isCommand = False
        try:
            command_line = attributes['command_line']
            for cmd in commands:
                if cmd in command_line:
                    isCommand = True
        except:
            pass
        
        if isImage and isCommand:
            T1135_e1.append((u, v, attributes))
            #T1083_e1.append((u, v, attributes))
            #print(attributes)
for item in T1135_e1:
    T1135_proc.add(item[0])

In [ ]:
len(T1135_proc)

In [ ]:
# T1040 Network Sniffing
# (process_name="tshark.exe" OR "windump.exe" OR "logman.exe" OR "tcpdump.exe" OR "wprui.exe" OR "wpr.exe")
# (process_name=""netsh.exe"" AND process_command_line=""*trace*start*capture=yes*"") OR
# process_name=""tshark.exe"" OR process_name=""wireshark.exe"")"
T1040_proc = set()
T1040_e1 = []
images = ['tshark.exe', 'windump.exe', 'logman.exe', 'tcpdump.exe', 'wprui.exe', 'wpr.exe', 'netsh.exe', 'wireshark.exe']
for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    #print(attributes)
    if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
        try:
            image_path = attributes['image_path']
            for img in images:
                if img in image_path:
                    T1040_proc.add(u)
                    T1040_e1.append((u, v, attributes))
        except:
            pass
                    
                    
print(len(T1040_proc))

In [ ]:
# T1201 Password policy discovery
# (process_command_line="*net* accounts*" OR "*net* accounts \/domain*")
T1201_proc = set()
T1201_e1 = []
commands = ['net.*accounts']
for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    #print(attributes)
    if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
        try:
            command_line = attributes['command_line']
            for cmd in commands:
                if bool(re.search(cmd, command_line)):
                    T1201_proc.add(u)
                    T1201_e1.append((u, v, attributes))
                    break
        except:
            pass
                    
                    
print(len(T1201_proc))

In [ ]:
# T1069 - permission group Discovery 
# process_name="net.exe" AND, 
# process_command_line="*net* user*" OR "*net* group*" OR "*net* localgroup*" OR "*get-localgroup*" OR "*get-ADPrinicipalGroupMembership*"
T1069_proc = set()
T1069_e1 = []
T1069_e2 = []
commands = ["net.* user", "net.* group", "net.* localgroup", "get-localgroup", "get-ADPrinicipalGroupMembership"]
for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    #print(attributes)
    if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
        try:
            command_line = attributes['command_line']
            for cmd in commands:
                if bool(re.search(cmd, command_line)):
                    T1069_e1.append((u, v, attributes))
                    break
        except:
            pass

for item in T1069_e1:
    for dst, attrs in G[item[0]].items():
        v = dst
        for key, attr in attrs.items():
            if G.nodes[v]["oType"] == "FLOW" and attr['attributes']['action'] == "MESSAGE":
                T1069_proc.add(item[0])
                T1069_e2.append((item[0], v, attr))
                
                
print(len(T1069_proc))
print(len(T1069_e1))
print(len(T1069_e2))

In [ ]:
# T1057 Process Discovery 
# process_name="tasklist.exe" OR process_command_line="*Get-Process*"
T1057_proc = set()
T1057_e1 = []
for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    #print(attributes)
    if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
        isCommand = False
        try:
            command_line = attributes['command_line']
            if "get-process" in command_line or "tasklist" in command_line:
                isCommand = True
        except:
            isCommand = False

        isImage = False
        try:
            image_path = attributes['image_path']
            if "tasklist.exe" in image_path:
                isImage = True
        except:
            isImage = False
    if isCommand or isImage:
        T1057_proc.add(u)
        T1057_e1.append((u, v, attributes))
        #print(attributes)
print(len(T1057_e1))

In [ ]:
# T1012 Query Registry
# Sysmon3- Network MESSAGE	- (process_name="reg.exe" AND process_command_line="*reg* query*")
# Sysmon1- process START	- (process_name="reg.exe" AND process_command_line="*reg* query*")
T1012_proc = set()
T1012_e1 = []
for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    #print(attributes)
    if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
        isCommand = False
        try:
            command_line = attributes['command_line']
            if bool(re.search("reg.*query", command_line)):
                isCommand = True
        except:
            isCommand = False

        isImage = False
        try:
            image_path = attributes['image_path']
            if "reg.exe" in image_path:
                isImage = True
        except:
            isImage = False
    if isCommand and isImage:
        T1012_proc.add(u)
        T1012_e1.append((u, v, attributes))
        #print(attributes)
print(len(T1012_e1))

In [ ]:
# [T1018] Remote System Discovery
T1018_proc = set()
T1018_e1 = []
T1018_e2 = []
for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    #print(attributes)
    if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
        isCommand = False
        try:
            command_line = attributes['command_line']
            if bool(re.search("net.*view", command_line)) or bool(re.search("ping.*", command_line)):
                #print("command")
                isCommand = True
        except:
            isCommand = False

        isImage = False
        try:
            image_path = attributes['image_path']
            if "net.exe" in image_path or "ping.exe" in image_path:
                #print("path")
                isImage = True
        except:
            isImage = False
        if isCommand or isImage:
            #T1018_proc.add(u)
            T1018_e1.append((u, v, attributes))
            #print(attributes)
        
for item in T1018_e1:
    for dst, attrs in G[item[0]].items():
        v = dst
        for key, attr in attrs.items():
            if G.nodes[v]["oType"] == "FLOW" and attr['attributes']['action'] == "MESSAGE":
                T1018_proc.add(item[0])
                T1018_e2.append((item[0], v, attr))
                

print(len(T1018_proc))
print(len(T1018_e1))
print(len(T1018_e2))

In [ ]:
# T1063 Security Software Discovery
# (process_name="netsh.exe" OR process_name="reg.exe" OR process_name="tasklist.exe") AND 
# (process_command_line="*reg* query*" OR process_command_line="*tasklist *" OR process_command_line="*netsh*" OR process_command_line="*fltmc*|*findstr*")

image_dict = ["netsh.exe", "reg.exe", "tasklist.exe"]
cmd_dict = ["reg.*query", "tasklist.*", "netsh.*", "fltmc.*" , "findstr*"]
T1063_proc = set()
T1063_e1 = []
for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    #print(attributes)
    if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
        isCommand = False
        try:
            command_line = attributes['command_line']
            for cmd in cmd_dict:
                if bool(re.search(cmd, command_line)):
                    #print("command")
                    isCommand = True
        except:
            isCommand = False

        isImage = False
        try:
            image_path = attributes['image_path']
            if "netsh.exe" in image_path or "reg.exe" in image_path or "tasklist.exe" in image_path:
                #print("path")
                isImage = True
        except:
            isImage = False
        if isCommand and isImage:
            T1063_proc.add(u)
            T1063_e1.append((u, v, attributes))

print(len(T1063_proc))

In [ ]:
# T1082 System Information Discovery
# (process_name="sysinfo.exe") OR 
# (process_name="reg.exe" AND 
cmd=r"reg.*query HKLM\\SYSTEM\\CurrentControlSet\\Services\\Disk\\Enum"
T1082_proc = set()
T1082_e1 = []
for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    #print(attributes)
    if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
        isCommand = False
        try:
            image_path = attributes['image_path']
            if "sysinfo.exe" in image_path:
                isCommand = True
            elif "reg.exe" in image_path:
                command_line = attributes['command_line']
                if bool(re.search(cmd, command_line)):
                    #print("command")
                    isCommand = True
            else:
                isCommand = False
        except:
            isCommand = False
        if isCommand:
            T1082_proc.add(u)
            T1082_e1.append((u, v, attributes))

print(len(T1082_proc))
print(len(T1082_e1))

In [ ]:
# T1049 System Network Connections Discovery
# (process_name="net.exe" OR process_name="netstat.exe") AND 
# (process_command_line="*net* use*" OR ""*net* sessions*" OR "*net* file*" OR "*netstat*") OR
# process_command_line="*Get-NetTCPConnection*"
T1049_proc = set()
T1049_e1 = []
cmd_dict = ["net.*use", "net.*sessions", "net.*file", "netstat.*", "Get-NetTCPConnection.*"]
for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    #print(attributes)
    if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
        isImage = False
        try:
            image_path = attributes['image_path']
            if "net.exe" in image_path or "netstat.exe" in image_path:
                isImage = True
        except:
            isImage = False
        
        isCommand = False
        try:
            command_line = attributes['command_line']
            for cmd in cmd_dict:
                if bool(re.search(cmd, command_line)):
                    #print("command")
                    isCommand = True
        except:
            isCommand = False
            
        if isCommand and isImage:
            T1049_proc.add(u)
            T1049_e1.append((u, v, attributes))

print(len(T1049_proc))
print(len(T1049_e1))

In [ ]:
# T1007 System Service Discovery
# (process_name="net.exe", "tasklist.exe", "sc.exe","wmic.exe") AND 
# (process_command_line="net.*start", "tasklist \/svc.*", "sc.*query","wmic.*service where")

image_dict = ["net.exe", "tasklist.exe", "sc.exe","wmic.exe"]
cmd_dict = ["net.*start", "tasklist \/svc.*", "sc.*query","wmic.*service where"]
T1007_proc = set()
T1007_e1 = []

for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    #print(attributes)
    if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
        isImage = False
        try:
            image_path = attributes['image_path']
            for img in image_dict:
                if img in image_path:
                    isImage = True
        except:
            isImage = False
        
        isCommand = False
        try:
            command_line = attributes['command_line']
            for cmd in cmd_dict:
                if bool(re.search(cmd, command_line)):
                    #print("command")
                    isCommand = True
        except:
            isCommand = False
            
        if isCommand and isImage:
            T1007_proc.add(u)
            T1007_e1.append((u, v, attributes))

print(len(T1007_proc))
print(len(T1007_e1))

In [ ]:
# T1124 System Time Discovery
# (process_path="*\\net.exe" AND process_command_line="*net* time*")
# OR process_name="w32tm.exe" OR process_command_line="*Get-Date*"

T1124_proc = set()
T1124_e1 = []

for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    #print(attributes)
    if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
        isCand = False
        try:
            image_path = attributes['image_path']
            command_line = attributes['command_line']
            if 'net.exe' in image_path and bool(re.search('net.*time', command_line)):
                isCand = True
            elif "w32tm.exe" in image_path or "Get-Date" in command_line:
                isCand = True
            else:
                isCand = False
        except:
            isCand = False
        
        if isCand:
            T1124_proc.add(u)
            T1124_e1.append((u, v, attributes))

print(len(T1124_proc))
print(len(T1124_e1))

## Lateral Movement Techniques 
So far I have detection rules for following TTPs. Also look at the threathunting rule book of 12306Bro
### T1037 Logon Scripts
### T1075 Pass the Hash (still in github)
### T1076 Remote Desktop Protocol
### T1077 Windows Admin Shares 
### T1028 Windows Remote Management

In [ ]:
# T1037 Logon Scripts 
# Sysmon Event ID = 1 
# Windows Security Event ID = 4688
# process_command_line = "REG.*ADD.*HKCU\\Environment.*UserInitMprLogonScript.*"

T1037_e1 = []
T1037_proc = set()
for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    #print(attributes)
    if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
        isCand = False
        try:
            command_line = attributes['command_line']
            if bool(re.search('REG.*ADD.*HKCU.*Environment.*UserInitMpsLogonScript.*', command_line)):
                isCand = True
        except:
            isCand = False
        
        if isCand:
            T1037_proc.add(u)
            T1037_e1.append((u, v, attributes))

print(len(T1037_proc))
print(len(T1037_e1))

In [ ]:
# T1075 Pass the hash 
# Windows Security Event ID = 4624 AND 
# (sid == "NULL SID" OR "S-1-0-0") AND
# (Logon_Type="3") AND
# (Source_Network_Address != "*::1*") AND
# (Logon_Process="*NtLmSsp") AND 
# (Package_Name__NTLM_only_="*NTLM V2") AND
# (Key_Length="0") AND 
# (user != "*ANONYMOUS LOGON" OR Account_Name != "*ANONYMOUS LOGON"))

In [ ]:
# T1076 Remote Desktop Protocol

#Event 1
#Sysmon Event ID = 1
#Windows Security Event ID = 4688
#process_name = "tscon.exe" OR process_name="mstsc.exe"

#Event 2
#Sysmon Event ID = 3
#process_path = "*\\tscon.exe" OR process_name="mstsc.exe" OR dst_port=3389 
#initiated = true

#Event 3
#Sysmon Event_id = 12 OR 13 OR 14
#process_path = "C:\\Windows\\system32\\LogonUI.exe" OR
#registry_key_path = "*\\SOFTWARE\\Policies\\Microsoft\\Windows NT\\Terminal Services\\*"

T1076_proc = set()
T1076_e1 = []
T1076_e2 = []
T1076_e3 = []

for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    #print(attributes)
    if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
        try:
            image_path = attributes['image_path']
            if "tscon.exe" in image_path or "mstsc.exe" in image_path:
                T1076_proc.add(u)
                T1076_e1.append((u, v, attributes))
        except:
            pass

for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    #print(attributes)
    if G.nodes[v]["oType"] == "FLOW" and attributes['action'] == "MESSAGE":
        try:
            image_path = attributes['image_path']
            dst_port = attributes['dest_port']
            if "tscon.exe" in image_path or "mstsc.exe" in image_path or dest_port == '3389':
                T1076_proc.add(u)
                T1076_e2.append((u, v, attributes))
        except:
            pass

for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    #print(attributes)
    regActions = ['ADD', 'EDIT', 'REMOVE']
    if G.nodes[v]["oType"] == "REGISTRY" and attributes['action'] in regActions:
        try:
            image_path = attributes['image_path']
            registry_path = attributes['key']
            if "LogonUI.exe" in image_path or "Terminal Services" in registry_path:
                T1076_proc.add(u)
                T1076_e3.append((u, v, attributes))
                #print(attributes)
        except:
            pass

print(len(T1076_proc))
print(len(T1076_e1))
print(len(T1076_e2))
print(len(T1076_e3))

In [ ]:
# T1077 Windows Admin Shares
# Event 1
# Sysmon Event ID = 3
# process_name = "net.exe" AND
# process_command_line = "net.*use.*$" OR "net.*session.*$" OR "net.*file.*$"
cmd_dict1 = ['net.*use.*$', 'net.*session.*$', 'net.*file.*$']

# Event 2
# Sysmon Event ID = 1
# Windows Security Event ID = 4688
# process_name = "net.exe" AND process_command_line = "net.*share.*$"
cmd_dict2 = ['net.*share.*$']

# Event 3
# Sysmon Event ID = 1
# Windows Security Event ID = 4688
# process_name = "net.exe" OR process_name=powershell.exe) AND
# process_command_line = "net.*use.*$" OR "net.*session.*$" OR "net.*file.*$" OR "New-PSDrive.*root.*"
cmd_dict3 = ['net.*use.*$', 'net.*session.*$', 'net.*file.*$', 'New-PSDrive.*root.*']

T1077_proc = set()
T1077_e1 = []
T1077_e2 = []
T1077_e3 = []

for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    #print(attributes)
    if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
        try:
            image_path = attributes['image_path']
            if "net.exe" in image_path:
                command_line = attributes['command_line']
                for cmd in cmd_dict1:
                    if bool(re.search(cmd, command_line)):
                        T1077_proc.add(u)
                        T1077_e1.append((u, v, attributes))
        except:
            pass

for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    #print(attributes)
    if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
        try:
            image_path = attributes['image_path']
            if "net.exe" in image_path:
                command_line = attributes['command_line']
                for cmd in cmd_dict2:
                    if bool(re.search(cmd, command_line)):
                        T1077_proc.add(u)
                        T1077_e2.append((u, v, attributes))
        except:
            pass

for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    #print(attributes)
    if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
        try:
            image_path = attributes['image_path']
            if "net.exe" in image_path or "powershell.exe" in image_path:
                command_line = attributes['command_line']
                for cmd in cmd_dict3:
                    if bool(re.search(cmd, command_line)):
                        T1077_proc.add(u)
                        T1077_e3.append((u, v, attributes))
        except:
            pass

print(len(T1077_proc))
print(len(T1077_e1))
print(len(T1077_e2))
print(len(T1077_e3))

In [ ]:
# T1028 Windows Remote Management (WinRM)
# process_name = "wsmprovhost.exe" OR "winrm.cmd" OR 
# process_command_line = ".*Enable-PSRemoting -Force.*" OR ".*Invoke-Command -computer_name.*" OR "wmic.*node.*process call create.*"
T1028_proc = set()
T1028_e1 = []

cmd_dict = [".*Enable-PSRemoting -Force.*", ".*Invoke-Command -computer_name.*", "wmic.*node.*process call create.*"]
for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    #print(attributes)
    if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
        try:
            image_path = attributes['image_path']
            if "wsmprovhost.exe" in image_path or "winrm.cmd" in image_path:
                command_line = attributes['command_line']
                for cmd in cmd_dict:
                    if bool(re.search(cmd, command_line)):
                        T1028_proc.add(u)
                        T1028_e1.append((u, v, attributes))
        except:
            pass

print(len(T1028_proc))
print(len(T1028_e1))


In [ ]:
book = ["\\SOFTWARE\\Microsoft\\Windows\\CurrentVersion\\Authentication\\Credential Provider\\", 
            "\\SYSTEM\\CurrentControlSet\\Control\\Lsa\\", 
            "\\SYSTEM\\CurrentControlSet\\Control\\SecurityProviders\\SecurityProviders\\",
            "\\Control\\SecurityProviders\\WDigest\\"]
page = "\\SYSTEM\\CurrentControlSet\\"

for mypage in book:
    if page in mypage:
        print(mypage)

# Credential Access

The adversary is trying to steal account names and passwords. Credential Access consists of techniques for stealing credentials like account names and passwords. Techniques used to get credentials include keylogging or credential dumping. Using legitimate credentials can give adversaries access to systems, make them harder to detect, and provide the opportunity to create more accounts to help achieve their goals.

* T1003: Credential Dumping
* T1081: Credentials in File 
* T1214: Credentials in Registry
* T1187: Forced Authentication
* T1040: Network Sniffing 
* T1179: Hooking 

In general it was a very disappointing run in credential access. I didnot get any hit on any of the TTPs I checked. Maybe need to be more specific.

In [ ]:
# T1003 Credential DUmping 
# Event 1
# Sysmon Event ID = 10
# target_process_path = "C:\\Windows\\system32\\lsass.exe" AND
# process_granted_access = [0x1010 OR 0x1410 OR 0x147a OR 0x143a]
# process_call_trace = "C:\\Windows\\SYSTEM32\\ntdll.dll\* OR C:\\Windows\\system32\\KERNELBASE.dll* OR UNKNOWN(*)"
# NA: doesn't apply to ecar data format 

# Event 2
# Sysmon Event ID = 12 OR 13 OR 14
# process_path != "C:\\WINDOWS\\system32\\lsass.exe"
# registry_key_path = "*\\SOFTWARE\\Microsoft\\Windows\\CurrentVersion\\Authentication\\Credential Provider\\*" OR 
#                    "*\\SYSTEM\\CurrentControlSet\\Control\\Lsa\\*" OR
#                    "*\\SYSTEM\\CurrentControlSet\\Control\\SecurityProviders\\SecurityProviders\\*" OR
#                    "*\\Control\\SecurityProviders\\WDigest\\*" AND 
# registry_key_path != "*\\Lsa\\RestrictRemoteSamEventThrottlingWindow"
T1003_proc = set()
T1003_e1 = []

regkey_dict = ["\\SOFTWARE\\Microsoft\\Windows\\CurrentVersion\\Authentication\\Credential Provider\\", 
            "\\SYSTEM\\CurrentControlSet\\Control\\Lsa\\", 
            "\\SYSTEM\\CurrentControlSet\\Control\\SecurityProviders\\SecurityProviders\\",
            "\\Control\\SecurityProviders\\WDigest\\"]
regkey_excl = "\\Lsa\\RestrictRemoteSamEventThrottlingWindow"
for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    if G.nodes[v]["oType"] == "REGISTRY":
        try:
            #print(attributes)
            registry_key_path = re.escape(attributes['key'])
            #print(registry_key_path)
            for mykey in regkey_dict:
                if mykey in registry_key_path and regkey_excl not in registry_key_path:
                    T1003_proc.add(u)
                    T1003_e1.append((u,v,attributes))
        except:
            pass

#print(len(T1003_proc))
#print(len(T1003_e1))


# Event 3
# Sysmon Event ID = 7
# driver_loaded = "C:\\Windows\\System32\\samlib.dll" OR
#                 "C:\\Windows\\System32\\WinSCard.dll" OR
#                 "C:\\Windows\\System32\\cryptdll.dll" OR
#                 "C:\\Windows\\System32\\hid.dll" OR
#                 "C:\\Windows\\System32\\vaultcli.dll"
# process_path != "*\\Sysmon.exe" AND process_path!="*\\svchost.exe" AND process_path!="*\\logonui.exe"

module_dict = ["samlib.dll", "WinSCard.dll", "cryptdll.dll", "hid.dll", "vaultcli.dll"]
image_dict = ["Sysmon.exe", "svchost.exe", "logonui.exe"]

T1003_e2 = []
for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    if G.nodes[v]["oType"] == "MODULE" and attributes['action'] == "LOAD":
        try:
            #print(attributes)
            module_path = attributes['module_path']
            image_path = attributes['image_path']
            isModule = False
            for mydll in module_dict:
                if mydll in module_path:
                    isModule = True
            
            for img in image_dict:
                if img in image_path:
                    isModule = False
                    break
            if isModule:
                T1003_proc.add(u)
                T1003_e2.append((u,v,attributes))
        except:
            pass

#print(len(T1003_proc))
#print(len(T1003_e2))


# Event 4
# Sysmon Event ID = 1
# Windows Security Event ID = 4688
# process_name=reg.exe OR
# process_command_line = "*save*HKLM\\sam*" OR "*save*HKLM\\system*"
cmd_reg = ["save.*HKLM.*sam", "save.*HKLM.*system"]
T1003_e3 = []

for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
        try:
            #print(attributes)
            command_line = attributes['command_line']
            image_path = attributes['image_path']
            isImage = False
            if "reg.exe" in image_path:
                isImage = True
            isCommand = False
            for cmd in cmd_reg:
                if bool(re.search(cmd, command_line)):
                    #print(attributes)
                    isCommand = True
            if isImage and isCommand:
                T1003_proc.add(u)
                T1003_e3.append((u, v, attributes))
            
        except:
            pass

#print(len(T1003_proc))
#print(len(T1003_e3))


# Event 5
# Sysmon Event ID = 1
# Windows Security Event ID = 4688
# process_command_line = "*Invoke-Mimikatz -DumpCreds*" OR "gsecdump* -a" OR "wce* -o" OR 
#                        "procdump* -ma lsass.exe*" OR "ntdsutil*ac i ntds*ifm*create full"
cmd_reg = ["Involve-Mimikatz -DumpCreds.*", "gsecdump.*-a", "wce.*-o", "procdump.*-ma lsass.exe", "ntdsutil.*ac i ntds.*ifm.*create full"]
T1003_e4 = []

for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
        try:
            #print(attributes)
            command_line = attributes['command_line']
            
            isCommand = False
            for cmd in cmd_reg:
                if bool(re.search(cmd, command_line)):
                    #print(attributes)
                    isCommand = True
            if isCommand:
                T1003_proc.add(u)
                T1003_e4.append((u, v, attributes))
            
        except:
            pass

print(len(T1003_proc))
print(len(T1003_e1))
print(len(T1003_e2))
print(len(T1003_e3))
print(len(T1003_e4))


In [ ]:
# T1081: Credentials in File
# process_command_line = "*findstr* /si pass*" OR "*select-string -Pattern pass*" OR "*list vdir*/text:password*"
cmd_reg = ["findstr.*si pass", "select-string -Pattern pass", "list vdir.*text:password"]

T1081_proc = set()
T1081_e1 = []
for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
        try:
            #print(attributes)
            command_line = attributes['command_line']
            
            isCommand = False
            for cmd in cmd_reg:
                if bool(re.search(cmd, command_line)):
                    #print(attributes)
                    isCommand = True
            if isCommand:
                T1081_proc.add(u)
                T1081_e1.append((u, v, attributes))
        
        except:
            pass

print(len(T1081_proc))
print(len(T1081_e1))

In [ ]:
# T1214: Credentials in Registry
#Sysmon Event ID = 1
#Windows Security Event ID = 4688
# process_command_line = "*reg* query HKLM \/f password \/t REG_SZ \/s*" OR
#                       "reg* query HKCU \/f password \/t REG_SZ \/s" OR 
#                       "*Get-UnattendedInstallFile*" OR
#                       "*Get-Webconfig*" OR
#                       "*Get-ApplicationHost*" OR
#                       "*Get-SiteListPassword*" OR
#                       "*Get-CachedGPPPassword*" OR
#                       "*Get-RegistryAutoLogon*"

cmd_reg = ["reg.* query HKLM \/f password \/t REG_SZ \/s", "reg.*query HKCU \/f password \/t REG_SZ \/s",
           "Get-UnattendedInstallFile", "Get-Webconfig", "Get-ApplicationHost", "Get-SiteListPassword",
          "Get-CachedGPPPassword", "Get-RegistryAutoLogon"]
T1214_proc = set()
T1214_e1 = []
for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
        try:
            #print(attributes)
            command_line = attributes['command_line']
            
            isCommand = False
            for cmd in cmd_reg:
                if bool(re.search(cmd, command_line)):
                    #print(attributes)
                    isCommand = True
            if isCommand:
                T1214_proc.add(u)
                T1214_e1.append((u, v, attributes))
        
        except:
            pass
        
print(len(T1214_proc))
print(len(T1214_e1))

In [ ]:
# T1187: Forced Authentication
# `sysmon` event_id=11 (file_path="*.lnk" OR file_path="*.scf")

extensions = [".*\.lnk", ".*\.scf"]
T1187_proc = set()
T1187_e1 = []
for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    #print(attributes)
    if G.nodes[v]["oType"] == "FILE":# and attributes['action'] == "CREATE":
        print(attributes)
        try:
            print(attributes)
            file_name = attributes['file_path']
            #print(file_name)
            isCommand = False
            for ext in extensions:
                if bool(re.search(ext, file_name)):
                    #print(attributes)
                    isCommand = True
            if isCommand:
                T1187_proc.add(u)
                T1187_e1.append((u, v, attributes))
        
        except:
            pass

print(len(T1187_proc))
print(len(T1187_e1))

In [ ]:
# T1040: Network Sniffing
# Event 1 
# Sysmon Event ID = 1
# Windows Security Event ID = 4688
# process_name = ["tshark.exe", "windump.exe", "logman.exe", "tcpdump.exe", "wprui.exe", "wpr.exe"]
def searchT1040(T1040_proc, T1040_events):
    T1040_e1 = []
    T1040_e2 = []


    image_dict = ["tshark.exe", "windump.exe", "logman.exe", "tcpdump.exe", "wprui.exe", "wpr.exe"]
    for edge in G.edges:
        u = edge[0]
        v = edge[1]
        attributes = G.edges[edge]['attributes']
        if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
            try:
                #print(attributes)
                image_path = attributes['image_path']
                isPath = False
                for img in image_dict:
                    if img in image_path:
                        #print(attributes)
                        isPath = True
                if isPath:
                    T1040_proc.add(u)
                    T1040_e1.append((u, v, attributes))

            except:
                pass

    # Event 2
    # Sysmon Event ID = 1
    # Windows Security Event ID = 4688
    # (process_name = "netsh.exe" AND process_command_line = "*trace*start*capture=yes*") OR 
    # process_name = "tshark.exe" OR process_name = "wireshark.exe")\
    for edge in G.edges:
        u = edge[0]
        v = edge[1]
        attributes = G.edges[edge]['attributes']
        if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
            try:
                #print(attributes)
                command_line = attributes['command_line']
                image_path = attributes['image_path']

                if "netsh.exe" in image_path and bool(re.search("trace.*start.*capture=yes", command_line)):
                    T1040_proc.add(u)
                    T1040_e2.append((u, v, attributes))
                elif "tshark.exe" in image_path or "wireshark.exe" in image_path:
                    T1040_proc.add(u)
                    T1040_e2.append((u, v, attributes))
                else:
                    pass
            except:
                pass

    T1040_events.append(T1040_e1)
    T1040_events.append(T1040_e2)
#     print(len(T1040_proc))
#     print(len(T1040_e1))
#     print(len(T1040_e2))

In [ ]:
# T1179: Hooking
# process_name = "mavinject.exe" OR
# process_command_line = "*/INJECTRUNNING*"
def searchT1179(T1179_proc, T1179_events):
    T1179_e1 = []
    for edge in G.edges:
        u = edge[0]
        v = edge[1]
        attributes = G.edges[edge]['attributes']
        if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
            try:
                #print(attributes)
                command_line = attributes['command_line']
                image_path = attributes['image_path']
                if "mavinject.exe" in image_path or "INJECTRUNNING" in command_line:
                    T1179_proc.add(u)
                    T1179_e1.append((u, v, attributes))
            except:
                pass

    T1179_events.append(T1179_e1)
#     print(len(T1179_proc))
#     print(len(T1179_e1))

# Defense Evasion

The adversary is trying to avoid being detected. Defense Evasion consists of techniques that adversaries use to avoid detection throughout their compromise. Techniques used for defense evasion include uninstalling/disabling security software or obfuscating/encrypting data and scripts. Adversaries also leverage and abuse trusted processes to hide and masquerade their malware. Other tactics’ techniques are cross-listed here when those techniques include the added benefit of subverting defenses.

* T1197: BITS Jobs
* T1088: Bypass User Account Control
* T1146: Clear Command History
* T1223: Compiled HTML File
* T1122: Component Object Model Hijacking
* T1196: Control Panel Items
* T1140: Deobfuscate/Decode Files or Information
* T1089: Disable Security Tools
* T1073: DLL Side-Loading
* T1183: Image File Execution Options Injection
* T1054: Indicator Blocking
* T1070: Indicator Removal on Host
* T1036: Masquerading
* T1112: Modify Registry
* T1170: MSHTA
* T1126: Network Share Connection Removal

In [ ]:
# T1197: BITS Jobs 
# Event 1
# Sysmon Event ID = 3 (MESSAGE the FLOW)
# process_name = "bitsadmin.exe"
T1197_proc = set()
T1197_e1 = []
for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    if G.nodes[v]["oType"] == "FLOW" and attributes['action'] == "MESSAGE":
        try:
            #print(attributes)
            image_path = attributes['image_path']
            if "bitsadmin.exe" in image_path:
                T1197_proc.add(u)
                T1197_e1.append((u, v, attributes))
        except:
            pass


# Event 2
# Sysmon Event ID = 1
# Windows Security Event ID = 4688
# process_name = "bitsadmin.exe" OR
# process_command_line = "*Start-BitsTransfer*"
T1197_e2 = []
for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
        try:
            #print(attributes)
            image_path = attributes['image_path']
            command_line = attributes['command_line']
            if "bitsadmin.exe" in image_path or "Start-BitsTransfer" in command_line:
                T1197_proc.add(u)
                T1197_e2.append((u, v, attributes))
        except:
            pass

print(len(T1197_proc))
print(len(T1197_e1))
print(len(T1197_e2))


In [ ]:
# T1088: Bypass User Account Control
# Event 1
# Sysmon Event ID = 1
# Windows Security Event ID = 4688
# process_parent_path = ["*\\eventvwr.exe", "*\\fodhelper.exe"]
def searchT1088(T1088_proc, T1088_events):
    T1088_e1 = []
    T1088_e2 = []

    parent_paths = ["eventvwr.exe", "fodhelper.exe"]
    for edge in G.edges:
        u = edge[0]
        v = edge[1]
        attributes = G.edges[edge]['attributes']
        if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
            try:
                #print(attributes)
                parent_path = attributes['parent_image_path']
                if "eventvwr.exe" in parent_path or "fodhelper.exe" in parent_path:
                    T1088_proc.add(u)
                    T1088_e1.append((u, v, attributes))
            except:
                pass


    # Event 2
    # Sysmon Event ID = 12, 13, or 14
    # registry_key_path \in ["*\\mscfile\\shell\\open\\command\\*", "*\\ms-settings\\shell\\open\\command\\*"] AND
    # sid not in ["S-1-5-18", "S-1-5-19", "S-1-5-20"]
    registry_keys = ["\\mscfile\\shell\\open\\command\\", "\\ms-settings\\shell\\open\\command\\"]
    benign_sids = ["S-1-5-18", "S-1-5-19", "S-1-5-20"]
    for edge in G.edges:
        u = edge[0]
        v = edge[1]
        attributes = G.edges[edge]['attributes']
        if G.nodes[v]["oType"] == "REGISTRY":# and attributes['action'] == "OPEN":
            try:
                #print(attributes)
                registry_path = attributes['key']
                sid = attributes['sid']
                for mykey in registry_keys:
                    if mykey in registry_path and sid not in benign_sids:
                        T1088_proc.add(u)
                        T1088_e2.append((u, v, attributes))
            except:
                pass

    T1088_events.append(T1088_e1)
    T1088_events.append(T1088_e2)
#     print(len(T1088_proc))
#     print(len(T1088_e1))
#     print(len(T1088_e2))

In [ ]:
# T1146 Clear Command History 
# process_command_line = "*rm (Get-PSReadlineOption).HistorySavePath*" OR,
#                        "*del (Get-PSReadlineOption).HistorySavePath*" OR,
#                        "*Set-PSReadlineOption –HistorySaveStyle SaveNothing*" OR,
#                        "*Remove-Item (Get-PSReadlineOption).HistorySavePath*"

cmd_dict = ["rm (Get-PSReadlineOption).HistorySavePath", "del (Get-PSReadlineOption).HistorySavePath", 
            "Set-PSReadlineOption –HistorySaveStyle SaveNothing", "Remove-Item (Get-PSReadlineOption).HistorySavePath"]
T1146_proc = set()
T1146_e1 = []
for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
        try:
            #print(attributes)
            command_line = attributes['command_line']
            # print(command_line)
            for cmd in cmd_dict:
                if cmd in command_line:
                    T1146_proc.add(u)
                    T1146_e1.append((u, v, attributes))
        except:
            pass

print(len(T1146_proc))
print(len(T1146_e1))

In [ ]:
# T1191: CMSTP
# Sysmon Event ID = 1
# Windows Security Event ID = 4688
# process_name = "CMSTP.exe"
T1191_proc = set()
T1191_e1 = []
for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
        try:
            image_path = attributes['image_path']
            # print(command_line)
            if "CMSTP.exe" in image_path:
                T1191_proc.add(u)
                T1191_e1.append((u, v, attributes))
        except:
            pass

print(len(T1191_proc))
print(len(T1191_e1))

In [ ]:
# T1223: Compiled HTML File.md
# Sysmon Event ID = 1
# Windows Security Event ID = 4688
# process_name = "hh.exe"
T1223_proc = set()
T1223_e1 = []
for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
        try:
            image_path = attributes['image_path']
            # print(command_line)
            if "hh.exe" in image_path:
                T1223_proc.add(u)
                T1223_e1.append((u, v, attributes))
        except:
            pass

print(len(T1223_proc))
print(len(T1223_e1))

In [ ]:
# T1122 Component Object Model Hijacking
T1122_proc = set()
T1122_e1 = []

mykey = "\\Software\\Classes\\CLSID\\"
for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    if G.nodes[v]["oType"] == "REGISTRY":# and attributes['action'] == "OPEN":
        try:
            #print(attributes)
            registry_path = attributes['key']
            if mykey in registry_path:
                T1122_proc.add(u)
                T1122_e1.append((u, v, attributes))
        except:
            pass

print(len(T1122_proc))
print(len(T1122_e1))

In [ ]:
# T1196 Control Panel Items
# Event 1
# Sysmon Event ID = 1
# Windows Security Event ID = 4688
# process_command_line = "*control* \/name*" OR "rundll32* shell32.dll, Control_RunDLL"
cmd_dict = ["control.*\/name", "rundll32.*shell32.dll,\ Control_RunDLL"]

T1196_proc = set()
T1196_e1 = []
T1196_e2 = []

for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
        try:
            command_line = attributes['command_line']
            for cmd in cmd_dict:
                if bool(re.search(cmd, command_line)):
                    T1196_proc.add(u)
                    T1196_e1.append((u, v, attributes))
        except:
            pass


# Event 2
# Sysmon Event ID = 12 OR 13 OR 14
# registry_key_path = "*\\SOFTWARE\\Microsoft\\Windows\\CurrentVersion\\Explorer\\ControlPanel\\NameSpace*" OR 
#                     "*\\Software\\Microsoft\\Windows\\CurrentVersion\\Controls Folder\\*\\Shellex\\PropertySheetHandlers\\*" OR
#                     "*\\Software\\Microsoft\\Windows\\CurrentVersion\\Control Panel\\*"

registry_keys = ["\\SOFTWARE\\Microsoft\\Windows\\CurrentVersion\\Explorer\\ControlPanel\\NameSpace", 
                 "\\Software\\Microsoft\\Windows\\CurrentVersion\\Controls Folder\\.*\\Shellex\\PropertySheetHandlers",
                 "\\Software\\Microsoft\\Windows\\CurrentVersion\\Control Panel\\"]

for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    if G.nodes[v]["oType"] == "REGISTRY":
        try:
            registry_path = re.escape(attributes['key'])
            #print(registry_path)
            for mykey in registry_keys:
                if mykey in registry_path:
                    T1196_proc.add(u)
                    T1196_e2.append((u, v, attributes))
        except:
            pass

print(len(T1196_proc))
print(len(T1196_e1))
print(len(T1196_e2))

In [ ]:
# T1140: Deobfuscate/Decode Files or Information
# process_name = "certutil.exe" AND
# process_command_line = "*decode*"
T1140_proc = set()
T1140_e1 = []
for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
        try:
            command_line = attributes['command_line']
            image_path = attributes['image_path']
            if "decode" in command_line and "certutil.exe" in image_path:
                T1140_proc.add(u)
                T1140_e1.append((u, v, attributes))
        except:
            pass
        
print(len(T1140_proc))
print(len(T1140_e1))

In [ ]:
# T1089: Disable Security Tools
# process_name = ["net.exe" OR "sc.exe"]
# cmdline = "stop"
T1089_proc = set()
T1089_e1 = []

for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
        try:
            command_line = attributes['command_line']
            image_path = attributes['image_path']
            if ("net.exe" in image_path or "sc.exe" in image_path) and "stop" in command_line:
                T1089_proc.add(u)
                T1089_e1.append((u, v, attributes))
        except:
            pass
        
print(len(T1089_proc))
print(len(T1089_e1))

In [ ]:
# T1073 DLL Side-Loading
# Event 1
# Sysmon Event ID = 7 
# driver_loaded = "*\\System.Management.Automation.ni.dll" OR
#                 "*\\System.Management.Automation.dll" OR 
#                 "*\\PowerShdll.dll"
# process_name != "powershell.exe" OR process_name != "powershellise.exe"

module_dict = ["System.Management.Automation.ni.dll", "System.Management.Automation.dll", "PowerShdll.dll"]
image_dict = ["powershell.exe", "powershellise.exe"]

T1073_proc = set()
T1073_e1 = []
T1073_e2 = []
for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    if G.nodes[v]["oType"] == "MODULE" and attributes['action'] == "LOAD":
        try:
            #print(attributes)
            module_path = attributes['module_path']
            image_path = attributes['image_path']
            isModule = False
            for mydll in module_dict:
                if mydll in module_path:
                    isModule = True
            
            for img in image_dict:
                if img in image_path:
                    isModule = False
                    break
            if isModule:
                T1073_proc.add(u)
                T1073_e1.append((u,v,attributes))
        except:
            pass


# Event 2
# Sysmon Event ID = 7 
# driver_loaded = "wmiutils.dll"
# process_path != "C:\\Windows\\"
module_dict = ["wmiutils.dll"]
image_dict = ["\\\\Windows\\\\.*"]
for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    if G.nodes[v]["oType"] == "MODULE" and attributes['action'] == "LOAD":
        try:
            #print(attributes)
            module_path = attributes['module_path']
            image_path = re.escape(attributes['image_path'])
            isModule = False
            for mydll in module_dict:
                if mydll in module_path:
                    # print(attributes)
                    isModule = True
            
            for img in image_dict:
                if bool(re.search(img, image_path)):
                    # print(image_path)
                    isModule = False
                    break
            if isModule:
                T1073_proc.add(u)
                T1073_e2.append((u,v,attributes))
        except:
            pass
        
print(len(T1073_proc))
print(len(T1073_e1))
print(len(T1073_e2))

In [ ]:
# T1107 File Deletion
# process_command_line = ["*remove-item*", "vssadmin*Delete Shadows /All /Q*", 
#                         "*wmic*shadowcopy delete*", "*wbdadmin* delete catalog -q*", 
#                         "*bcdedit*bootstatuspolicy ignoreallfailures*", "*bcdedit*recoveryenabled no*"

T1107_proc = set()
T1107_e1 = []
cmd_dict = ["remove-item", "vssadmin.*Delete Shadows /All /Q", "wmic.*shadowcopy delete", "wbdadmin.*delete catalog -q",
            "bcdedit.*bootstatuspolicy ignoreallfailures", "bcdedit.*recoveryenabled no"]

for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
        try:
            command_line = attributes['command_line']
            for cmd in cmd_dict:
                if bool(re.search(cmd, command_line)):
                    T1107_proc.add(u)
                    T1107_e1.append((u, v, attributes))
        except:
            pass
        
print(len(T1107_proc))
print(len(T1107_e1))

In [ ]:
# T1158 Hidden Files and Directory 
# process_name = "attrib.exe" AND
# process_command_line = ["*+h*", "*+s*"]
T1158_proc = set()
T1158_e1 = []
T1158_e2 = []

for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
        try:
            image_path = attributes['image_path']
            command_line = attributes['command_line']
            if "attrib.exe" in image_path:
                print(image_path)
                if bool(re.search(".*\+h.*", command_line)) or bool(re.search(".*\+s.*", command_line)):
                    T1158_proc.add(u)
                    T1158_e1.append((u, v, attributes))
        except:
            pass

# Event 2
# Sysmon Event ID = 1
# Windows Security Event ID = 4688
# process_path = "*\\VolumeShadowCopy*\\*" OR "*\\VolumeShadowCopy*\\*"
for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
        try:
            image_path = attributes['image_path']
            command_line = attributes['command_line']
            if "VolumeShadowCopy" in image_path:
                T1158_proc.add(u)
                T1158_e2.append((u, v, attributes))
        except:
            pass

print(len(T1158_proc))
print(len(T1158_e1))
print(len(T1158_e2))

In [ ]:
# T1183 Image File Execution Options Injection
# registry_key_path = "*\\Software\\Microsoft\\Windows NT\\CurrentVersion\\Image File Execution Options\\*" OR
#                     "*\\Wow6432Node\\Microsoft\\Windows NT\\CurrentVersion\\Image File Execution Options\\*"

T1183_proc = set()
T1183_e1 = []
registry_keys = ["\\Software\\Microsoft\\Windows NT\\CurrentVersion\\Image File Execution Options\\",
                 "\\Wow6432Node\\Microsoft\\Windows NT\\CurrentVersion\\Image File Execution Options\\"]

for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    if G.nodes[v]["oType"] == "REGISTRY":
        try:
            registry_path = re.escape(attributes['key'])
            #print(registry_path)
            for mykey in registry_keys:
                if mykey in registry_path:
                    T1183_proc.add(u)
                    T1183_e1.append((u, v, attributes))
        except:
            pass

print(len(T1183_proc))
print(len(T1183_e1))

In [ ]:
# T1054: Indicator Blocking
# Event 1
# Sysmon Event ID = 12 or 13 or 14
# registry_key_path = "HKLM\\System\\CurrentControlSet\\Services\\SysmonDrv\\*" OR
#                     "HKLM\\System\\CurrentControlSet\\Services\\Sysmon\\*" OR
#                     "HKLM\\System\\CurrentControlSet\\Services\\Sysmon64\\*"
# process_name != "Sysmon64.exe" or "Sysmon.exe"
T1054_proc = set()
T1054_e1 = []
T1054_e2 = []
registry_keys = ["HKLM\\System\\CurrentControlSet\\Services\\SysmonDrv\\", 
                 "HKLM\\System\\CurrentControlSet\\Services\\Sysmon\\",
                 "HKLM\\System\\CurrentControlSet\\Services\\Sysmon64\\"]
image_dict = ["Sysmon64.exe", "Sysmon.exe"]

for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    if G.nodes[v]["oType"] == "REGISTRY":
        try:
            registry_path = re.escape(attributes['key'])
            image_path = attrubutes['image_path']
            isCand = False
            #print(registry_path)
            for mykey in registry_keys:
                if mykey in registry_path:
                    isCand = True
            for img in image_dict:
                if img in image_path:
                    isCand = False
            if isCand:
                T1054_proc.add(u)
                T1054_e1.append((u, v, attributes))
        except:
            pass

# Event 2
# Sysmon Event ID = 1
# Windows Security Event ID = 4688
# process_name = "fltmc.exe" OR
# process_command_line = "*fltmc*unload*"
for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
        try:
            image_path = attributes['image_path']
            command_line = attributes['command_line']
            if "fltmc.exe" in image_path or bool(re.search("fltmc.*unload", command_line)):
                T1054_proc.add(u)
                T1054_e2.append((u, v, attributes))
        except:
            pass

print(len(T1054_proc))
print(len(T1054_e1))
print(len(T1054_e2))

In [ ]:
# T1070: Indicator Removal on Host
# process_name = "wevtutil.exe" OR
# process_command_line = "*wevtutil* cl*"
T1070_proc = set()
T1070_e1 = []
for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
        try:
            image_path = attributes['image_path']
            command_line = attributes['command_line']
            if "wevtutil.exe" in image_path or bool(re.search("wevtutil.*cl", command_line)):
                T1070_proc.add(u)
                T1070_e1.append((u, v, attributes))
        except:
            pass

print(len(T1070_proc))
print(len(T1070_e1))

In [ ]:
# T1202: Indirect command execution 
# process_parent_name = "pcalua.exe" OR
# process_name = "pcalua.exe" OR "bash.exe" "forfiles.exe"
T1202_proc = set()
T1202_e1 = []
for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
        try:
            image_path = attributes['image_path']
            parent_path = attributes['parent_image_path']
            if "pcalua.exe" in parent_path or "pcalua.exe" in image_path or "bash.exe" in image_path or "forfiles.exe" in image_path:
                T1202_proc.add(u)
                T1202_e1.append((u, v, attributes))
        except:
            pass

print(len(T1202_proc))
print(len(T1202_e1))

In [ ]:
# T1130: Install Root Certificate
# process_name != "svchost.exe" AND
# registry_key_path = "*\\SOFTWARE\\Microsoft\\EnterpriseCertificates\\Root\\Certificates\\*" OR
#                     "*\\Microsoft\\SystemCertificates\\Root\\Certificates\\*")\
registry_keys = ["\\SOFTWARE\\Microsoft\\EnterpriseCertificates\\Root\\Certificates\\", 
                 "\\Microsoft\\SystemCertificates\\Root\\Certificates\\"]
T1130_proc = set()
T1130_e1 = []
for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    if G.nodes[v]["oType"] == "REGISTRY":
        try:
            image_path = attributes['image_path']
            registry_path = re.escape(attributes['key'])
            for reg in registry_keys:
                if reg in registry_path:
                    if "svchost.exe" not in image_path:
                        T1130_proc.add(u)
                        T1130_e1.append((u, v, attributes))
        except:
            pass

print(len(T1130_proc))
print(len(T1130_e1))

In [ ]:
# T1118: Install Util
# process_name = "InstallUtil.exe" OR
# process_command_line = "*\/logfile= \/LogToConsole=false \/U*"
T1118_proc = set()
T1118_e1 = []
for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
        try:
            image_path = attributes['image_path']
            command_line = attributes['command_line']
            if "InstallUtil.exe" in image_path or "\/logfile= \/LogToConsole=false \/U" in command_line:
                T1118_proc.add(u)
                T1118_e1.append((u, v, attributes))
        except:
            pass

print(len(T1118_proc))
print(len(T1118_e1))

In [ ]:
# T1036: Masquerading
T1036_proc = set()
T1036_e1 = []
T1036_e2 = []
T1036_e3 = []
T1036_e4 = []
# Event 1
# Sysmon Event ID = 1
# Windows Security Event ID = 4688
# process_name = ["*.doc.*", "*.docx.*", "*.xls.*", "*.xlsx.*", "*.pdf.*", "*.rtf.*", "*.jpg.*", "*.png.*", "*.jpeg.*", "*.zip.*", "*.rar.*", "*.ppt.*", "*.pptx.*"]
image_regex = [".*\.doc.*", ".*\.docx.*", ".*\.xls.*", ".*\.xlsx.*", ".*\.pdf.*", ".*\.rtf.*", ".*\.jpg.*", ".*\.png.*", ".*\.jpeg.*", ".*\.zip.*", ".*\.rar.*", ".*\.ppt.*", ".*\.pptx.*"]
for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
        try:
            image_path = attributes['image_path']
            for ext in image_regex:
                if bool(re.search(ext, image_path)):
                    T1036_proc.add(u)
                    T1036_e1.append((u, v, attributes))
        except:
            pass

# Event 2
# Sysmon Event ID = 11
# file_path = ["*SysWOW64*", "*System32*", "*AppData*"] AND 
# file_name = ["*.exe", "*.dll", "*.bat", "*.com", "*.ps1", "*.py", "*.js", "*.vbs", "*.hta"]

path_dict = [".*SysWOW64.*", ".*System32.*", ".*AppData.*"] 
ext_dict = [".*\.exe", ".*\.dll", ".*\.bat", ".*\.com", ".*\.ps1", ".*\.py", ".*\.js", ".*\.vbs", ".*\.hta"]
for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    if G.nodes[v]["oType"] == "FILE" and attributes['action'] == "CREATE":
        try:
            file_path = attributes['file_path']
            isPath = False
            for path in path_dict:
                if bool(re.search(path, file_path)):
                    print(file_path)
                    isPath = True
            isExt = False
            for ext in ext_dict:
                if bool(re.search(ext, file_path)):
                    isExt = True
                    
            if isPath and isExt:
                T1036_proc.add(u)
                T1036_e2.append((u, v, attributes))
        except:
            pass


# Event 3
# Sysmon Event ID = 1
# Windows Security Event ID = 4688
# (process_name = "svchost.exe" AND process_parent_name != "services.exe") OR
# process_name = "scvhost.exe"
for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
        try:
            image_path = attributes['image_path']
            parent_path = attributes['parent_image_path']
            if "svchost.exe" in image_path and "services.exe" not in parent_path:
                T1036_proc.add(u)
                T1036_e3.append((u, v, attributes))
        except:
            pass


# Event 4
# Sysmon Event ID = 1
# Windows Security Event ID = 4688
# (process_name = "explorer.exe" AND process_parent_name != "userinit.exe")
for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
        try:
            image_path = attributes['image_path']
            parent_path = attributes['parent_image_path']
            if "explorer.exe" in image_path and "userinit.exe" not in parent_path:
                T1036_proc.add(u)
                T1036_e4.append((u, v, attributes))
        except:
            pass

print(len(T1036_proc))
print(len(T1036_e1))
print(len(T1036_e2))
print(len(T1036_e3))
print(len(T1036_e4))

In [ ]:
# T1112: Modify Registry
# (process_name = "reg.exe" AND process_command_line != ".*query.*")
T1112_proc = set()
T1112_e1 = []
for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
        try:
            image_path = attributes['image_path']
            command_line = attributes['command_line']
            if "reg.exe" in image_path and "query" not in command_line:
                T1112_proc.add(u)
                T1112_e1.append((u, v, attributes))
        except:
            pass

print(len(T1112_proc))
print(len(T1112_e1))


In [ ]:
# T1170: MSHTA
# Event 1
# Sysmon Event ID = 1
# Windows Security Event ID = 4688
# process_parent_path = "*\\mshta.exe" OR process_name = "mshta.exe"
T1170_proc = set()
T1170_e1 = []
T1170_e2 = []
T1170_e3 = []
for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
        try:
            image_path = attributes['image_path']
            parent_path = attributes['parent_image_path']
            if "mshta.exe" in image_path or "mshta.exe" in parent_path:
                T1170_proc.add(u)
                T1170_e1.append((u, v, attributes))
        except:
            pass

# Event 2
# Sysmon Event ID = 3
# process_parent_path = "*\\mshta.exe" OR process_path = "*\\mshta.exe"

for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    if G.nodes[v]["oType"] == "FLOW" and attributes['action'] == "MESSAGE":
        try:
            image_path = attributes['image_path']
            if "mshta.exe" in image_path:
                T1170_proc.add(u)
                T1170_e2.append((u, v, attributes))
        except:
            pass

# Event 3
# Sysmon Event ID = 11 or 15
# file_path = "*.hta"
for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    if G.nodes[v]["oType"] == "FILE" and attributes['action'] == "CREATE":
        try:
            file_path = attributes['file_path']
            if bool(re.search(".*\.hta", file_path)):
                T1170_proc.add(u)
                T1170_e3.append((u, v, attributes))
        except:
            pass

print(len(T1170_proc))
print(len(T1170_e1))
print(len(T1170_e2))
print(len(T1170_e3))

In [ ]:
# T1126 Network Share Connection Removal
# Sysmon Event ID = 1
# Windows Security Event ID = 4688
# (process_name = "net.exe" AND process_command_line = "*net* delete*") OR 
# process_command_line = "*Remove-SmbShare*" OR "*Remove-FileShare*"
T1126_proc = set()
T1126_e1 = []
for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
        try:
            image_path = attributes['image_path']
            command_line = attributes['command_line']
            if "net.exe" in image_path and bool(re.search("net.* delete", command_line)):
                T1126_proc.add(u)
                T1126_e1.append((u, v, attributes))
            elif "Remove-SmbShare" in command_line or "Remove-FileShare" in command_line:
                T1126_proc.add(u)
                T1126_e1.append((u, v, attributes))
            else:
                pass
        except:
            pass

print(len(T1126_proc))
print(len(T1126_e1))

In [ ]:
# T1096: NTFS File Attributes
# Sysmon Event ID = 1
# Windows Security Event ID = 4688
# process_name="fsutil.exe"
# proces_command_line="*usn*deletejournal*"
T1096_proc = set()
T1096_e1 = []
for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
        try:
            image_path = attributes['image_path']
            command_line = attributes['command_line']
            if "fsutil.exe" in image_path and bool(re.search("usn.*deletejournal", command_line)):
                T1096_proc.add(u)
                T1096_e1.append((u, v, attributes))
            else:
                pass
        except:
            pass

print(len(T1096_proc))
print(len(T1096_e1))

In [ ]:
# T1027: Obfuscated Files or Information
# Sysmon Event ID = 1
# Windows Security Event ID = 4688
# (process_name = "certutil.exe" AND process_command_line = "*encode*") OR
# process_command_line="*ToBase64String*"
T1027_proc = set()
T1027_e1 = []
for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
        try:
            image_path = attributes['image_path']
            command_line = attributes['command_line']
            if "certutil.exe" in image_path and "encode" in command_line:
                T1027_proc.add(u)
                T1027_e1.append((u, v, attributes))
            elif "ToBase64String" in command_line:
                T1027_proc.add(u)
                T1027_e1.append((u, v, attributes))
            else:
                pass
        except:
            pass

print(len(T1027_proc))
print(len(T1027_e1))

In [ ]:
# T1093: Process Hollowing
T1093_proc = set()
T1093_e1 = []
T1093_e2 = []
T1093_e3 = []

# Event 1
# Sysmon Event ID = 1
# Windows Security Event ID = 4688
# (process_name = "smss.exe" AND process_parent_name!="smss.exe") OR
# (process_name = "csrss.exe" AND (process_parent_name != "smss.exe" AND process_parent_name != "svchost.exe")) OR
# (process_name = "wininit.exe" AND process_parent_name != "smss.exe") OR 
# (process_name = "winlogon.exe" AND process_parent_name != "smss.exe") OR
# (process_name = "lsass.exe" AND parent_process_name != "wininit.exe") OR 
# (process_name = "LogonUI.exe" AND (process_parent_name != "winlogon.exe" AND process_parent_name!="wininit.exe")) OR 
# (process_name = "services.exe" AND process_parent_name != "wininit.exe") OR 
# (process_name = "spoolsv.exe" AND process_parent_name != "services.exe") OR
# (process_name = "taskhost.exe" AND (process_parent_name != "services.exe" AND process_parent_name != "svchost.exe")) OR
# (process_name = "taskhostw.exe" AND (process_parent_name != "services.exe" AND process_parent_name != "svchost.exe")) OR 
# (process_name = "userinit.exe" AND (process_parent_name != "dwm.exe" AND process_parent_name != "winlogon.exe"))
process_names = ["smss.exe", "csrss.exe", "wininit.exe", "winlogon.exe", "lsass.exe", "LogonUI.exe", "services.exe", 
                 "spoolsv.exe", "taskhost.exe", "taskhostw.exe", "userinit.exe"]
parent_names = [["smss.exe"], 
                 ["csrss.exe", "svchost.exe"],
                 ["smss.exe"], 
                 ["smss.exe"],
                 ["wininit.exe"], 
                 ["winlogon.exe", "wininit.exe"],
                 ["wininit.exe"],
                 ["services.exe"],
                 ["services.exe", "svchost.exe"],
                 ["services.exe", "svchost.exe"],
                 ["dwm.exe", "winlogon.exe"]]

for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
        try:
            image_path = attributes['image_path']
            parent_path = attributes['parent_image_path']
            for i in range(len(process_names)):
                img = process_names[i]
                #print(img)
                iscand = False
                if img in image_path:
                    isCand = True
                for parent_img in parent_names[i]:
                    if parent_img in parent_path:
                        isCand = False
                if isCand:
                    T1093_proc.add(u)
                    T1093_e1.append((u, v, attributes))
        except:
            pass

# Event 2
# Sysmon Event ID = 1
# Windows Security Event ID = 4688
# process_command_line = "" OR process_command_line = $$process_path$$

# Event 3
# Sysmon Event ID = 1
# Windows Security Event ID = 4688
# process_parent_name =["winword.exe" or "excel.exe" or "outlook.exe") AND
# process_command_line = "C:\\Program Files\\Microsoft Office\\*-enc*"

for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
        try:
            parent_path = attributes['parent_image_path']
            command_line = attributes['command_line']
            if "winword.exe" in parent_path or "excel.exe" in parent_path or "outlook.exe" in parent_path:
                if "C:\\Program Files\\Microsoft Office\\" in command_line and "enc" in command_line:
                    T1093_proc.add(u)
                    T1093_e3.append((u, v, attributes))
        except:
            pass

print(len(T1093_proc))
print(len(T1093_e1))
print(len(T1093_e2))
print(len(T1093_e3))

In [ ]:
# T1055: Process Injection
T1055_proc = set()
T1055_e1 = []
T1055_e2 = []
T1055_e3 = []
# Event 1
# Sysmon Event ID = 8 (REMOTE_CREATE)
# StartFunction = "*LoadLibrary*"


# Event 2
# Sysmon Event ID = 8 (REMOTE_CREATE)
# target_process_address = 0x*0B80

# Event 3
# Sysmon Event ID = 1
# Windows Security Event ID = 4688
# process_command_line = "*Invoke-DllInjection*" or "*C:\\windows\sysnative\\*"
for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
        try:
            parent_path = attributes['parent_image_path']
            command_line = re.escape(attributes['command_line'])
            if "Invoke-DllInjection" in command_line or "C:\\Windows\\Sysnative" in command_line:
                T1055_proc.add(u)
                T1055_e3.append((u, v, attributes))
        except:
            pass

print(len(T1055_proc))
print(len(T1055_e1))
print(len(T1055_e2))
print(len(T1055_e3))

# TODO: Didn't see the StartFunction or target_process_address or command line in CREATE_REMOTE 

In [ ]:
# T1121: Regsvcs/ Regasm
# (process_name="regsvcs.exe" OR process_name="regasm.exe")
T1121_proc = set()
T1121_e1 = []

for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
        try:
            image_path = attributes['image_path']
            command_line = attributes['command_line']
            if "regsvcs.exe" in image_path or "regasm" in image_path:
                T1121_proc.add(u)
                T1121_e1.append((u, v, attributes))
        except:
            pass

print(len(T1121_proc))
print(len(T1121_e1))

In [ ]:
# T1117: Regsvr32 
T1117_proc = set()
T1117_e1 = []
T1117_e2 = []
# Event 1
# Sysmon Event ID = 3
# process_parent_path = "*\\regsvr32.exe" OR 
# process_path = "*\\regsvr32.exe"
for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
        try:
            image_path = attributes['image_path']
            parent_path = attributes['parent_image_path']
            if "regsvr32.exe" in image_path or "regsvr32.exe" in parent_path:
                T1117_proc.add(u)
                T1117_e1.append((u, v, attributes))
        except:
            pass

# Event 2
# Sysmon Event ID = 1
# Windows Security Event ID = 4688
# process_name = "regsvr32.exe" or "rundll32.exe" or "certutil.exe") OR
# process_command_line = "*scrobj.dll*"
image_dict = ["regsvr32.exe", "rundll32.exe", "certutil.exe"]
for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
        try:
            image_path = attributes['image_path']
            command_line = attributes['command_line']
            for img in image_dict:
                if img in image_path or "scrobj.dll" in command_line:
                    T1117_proc.add(u)
                    T1117_e2.append((u, v, attributes))
        except:
            pass
        
print(len(T1117_proc))
print(len(T1117_e1))
print(len(T1117_e2))

In [ ]:
# T1085: Rundll32
# process_parent_path = "*\\rundll32.exe" OR process_name="rundll32.exe"
T1085_proc = set()
T1085_e1 = []

for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
        try:
            image_path = attributes['image_path']
            parent_path = attributes['parent_image_path']
            if "rundll32.exe" in image_path or "rundll32.exe" in parent_path:
                T1085_proc.add(u)
                T1085_e1.append((u, v, attributes))
        except:
            pass

print(len(T1085_proc))
print(len(T1085_e1))

In [ ]:
# T1218: Signed Binary Proxy Execution
T1218_proc = set()
T1218_e1 = []
T1218_e2 = []
# Event 1
# Sysmon Event ID = 1
# Windows Security Event ID = 4688
# process_command_line = "*mavinject*\/injectrunning*" or "mavinject32*\/injectrunning*" or "*certutil*script\:http\[\:\]\/\/*" or
#                        "*certutil*script\:https\[\:\]\/\/*" or "*msiexec*http\[\:\]\/\/*" or "*msiexec*https\[\:\]\/\/*"

command_lines = ["mavinject.*\/injectrunning", "mavinject32.*\/injectrunning", "certutil.*script\:http\[\:\]\/\/", 
                  "certutil.*script\:https\[\:\]\/\/", "msiexec.*http\[\:\]\/\/", "msiexec.*https\[\:\]\/\/"]
for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
        try:
            image_path = attributes['image_path']
            command_line = attributes['command_line']
            for cmd in command_lines:
                if bool(re.search(cmd, command_line)):
                    T1218_proc.add(u)
                    T1218_e1.append((u, v, attributes))
        except:
            pass


# Event 2
# Sysmon Event ID = 3
# process_name = certutil.exe OR 
# process_command_line = "*certutil*script\:http\[\:\]\/\/*" OR 
# process_path = "*\\replace.exe"
for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    if G.nodes[v]["oType"] == "FLOW" and attributes['action'] == "MESSAGE":
        try:
            image_path = attributes['image_path']
            parent_path = attributes['parent_image_path']
            if "certutil.exe" in image_path or "replace.exe" in image_path or bool(re.search("certutil.*script\:http", command_line)):
                T1218_proc.add(u)
                T1218_e2.append((u, v, attributes))
        except:
            pass
print(len(T1218_proc))
print(len(T1218_e1))
print(len(T1218_e2))

In [ ]:
# T1216: Signed Script Proxy Execution
# Windows Security Event ID = 4688
# process_command_line = ["cscript.*script\:http\[\:\]\/\/", "wscript.*script\:http\[\:\]\/\/", "certutil.*script\:http\[\:\]\/\/", "*jjs*-scripting*"]
command_lines = ["cscript.*script\:http\[\:\]\/\/", "wscript.*script\:http\[\:\]\/\/", "certutil.*script\:http\[\:\]\/\/", "*jjs*-scripting*"]
T1216_proc = set()
T1216_e1 = []
for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
        try:
            image_path = attributes['image_path']
            command_line = attributes['command_line']
            for cmd in command_lines:
                if bool(re.search(cmd, command_line)):
                    T1216_proc.add(u)
                    T1216_e1.append((u, v, attributes))
        except:
            pass

print(len(T1216_proc))
print(len(T1216_e1))

In [ ]:
# T1127: Trusted Developer Utilities
T1127_proc = set()
T1127_e1 = []
T1127_e2 = []
# Event 1
# Sysmon Event ID = 1
# Windows Security Event ID = 4688
# process_name = "MSBuild.exe" or "msxsl.exe"
for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
        try:
            image_path = attributes['image_path']
            command_line = attributes['command_line']
            if "MSBuild.exe" in image_path or "msxsl.exe" in image_path:
                T1127_proc.add(u)
                T1127_e1.append((u, v, attributes))
        except:
            pass

# Event 2
# Sysmon Event ID = 11
# target_file_name = "\\AppData\\Local\\Microsoft\\CLR_v2.0.*\\UsageLogs\\"
#freg = "\\\\AppData\\\\Local\\\\Microsoft\\\\CLR_v4\.0.*\\\\UsageLogs"
freg = "\\\\AppData\\\\Local\\\\Microsoft\\\\CLR_v4\.0.*\\\\UsageLogs"

for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    if G.nodes[v]["oType"] == "FILE":# and attributes['action'] == "CREATE":
        try:
            file_path = re.escape(attributes['file_path'])
            print(file_path)
            if bool(re.search(freg, file_path)):
                T1127_proc.add(u)
                T1127_e2.append((u, v, attributes))
        except:
            pass

print(len(T1127_proc))
print(len(T1127_e1))
print(len(T1127_e2))

In [ ]:
allTypes = set()
for node in G.nodes:
    allTypes.add(G.nodes[node]["oType"])
print(allTypes)
## FILE is MISSING 

# Privilege Escalation

In [ ]:
# [T1053] Scheduled Task
def searchT1053(T1053_proc, T1053_events):
#    T1053_proc = set()
    T1053_e1 = []
    T1053_e2 = []
    # Event 1
    # Sysmon Event ID = 1
    # Windows Security Event ID = 4688
    # process_name = "taskeng.exe" or "schtasks.exe" or "svchost.exe"
    # process_parent_path != "C:\\Windows\\System32\\services.exe"

    for edge in G.edges:
        u = edge[0]
        v = edge[1]
        attributes = G.edges[edge]['attributes']
        if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
            try:
                image_path = attributes['image_path']
                parent_path = re.escape(attributes['parent_image_path'])
                if "taskeng.exe" in image_path or "schtasks.exe" in image_path or "svchost.exe" in image_path:
                    if "\\Windows\\System32\\services.exe" not in parent_path:
                        T1053_proc.add(u)
                        T1053_e1.append((u, v, attributes))
            except:
                pass

    # Event 2
    # Sysmon Event ID = 11
    # process_path != "C:\\WINDOWS\\system32\\svchost.exe" 
    # file_path = "C:\\Windows\\System32\\Tasks\\*" or "C:\\Windows\\Tasks\\*"
    fregs = ["\\Windows\\System32\\Tasks\\", "\\Windows\\Tasks\\"]
    for edge in G.edges:
        u = edge[0]
        v = edge[1]
        attributes = G.edges[edge]['attributes']
        if G.nodes[v]["oType"] == "FILE":# and attributes['action'] == "CREATE":
            try:
                file_path = re.escape(attributes['file_path'])
                for freg in fregs:
                    if freg in file_pathth:
                        T1053_proc.add(u)
                        T1053_e2.append((u, v, attributes))
            except:
                pass

    T1053_events.append(T1053_e1)
    T1053_events.append(T1053_e2)
#     print(len(T1053_proc))
#     print(len(T1053_e1))
#     print(len(T1053_e2))

In [ ]:
# T1013: Port Monitors
# Sysmon Event ID = 12 or 13 or 14
# registry_key_path = "*\\SYSTEM\CurrentControlSet\Control\Print\Monitors\\*"
def searchT1013(T1013_proc, T1013_events):
    T1013_e1 = []
    mykey = "\\SYSTEM\\CurrentControlSet\\Control\\Print\\Monitors\\"
    for edge in G.edges:
        u = edge[0]
        v = edge[1]
        attributes = G.edges[edge]['attributes']
        if G.nodes[v]["oType"] == "REGISTRY":# and attributes['action'] == "CREATE":
            try:
                registry_path = re.escape(attributes['key'])
                if mykey in registry_path:
                    T1013_proc.add(u)
                    T1013_e1.append((u, v, attributes))
            except:
                pass
    T1013_events.append(T1013_e1)
#     print(len(T1013_proc))
#     print(len(T1013_e1))

In [ ]:
# T1050 New Service 
# Sysmon Event ID = 1
# Windows Security Event ID = 4688
# process_name = "sc.exe" or "powershell.exe"or "cmd.exe" AND
# process_command_line = "*New-Service*BinaryPathName*" or "*sc*create*binpath*" or "*Get-WmiObject*Win32_Service*create*"
def searchT1050(T1050_proc, T1050_events):
    T1050_e1 = []
    
    image_dict = ["sc\.exe", "powershell\.exe", "cmd\.exe"]
    command_lines = ["New-Service.*BinaryPathName", "sc.*create.*binpath", "Get-WmiObject.*Win32_Service.*create"]
    for edge in G.edges:
        u = edge[0]
        v = edge[1]
        attributes = G.edges[edge]['attributes']
        if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
            try:
                image_path = attributes['image_path']
                command_line = attributes['command_line']
                isImage = False
                for img in image_dict:
                    if bool(re.search(img, image_path)):
                        isImage = True

                isCommand = False
                for cmd in command_lines:
                    if bool(re.search(cmd, command_line)):
                        isCommand = True

                if isImage and isCommand:
                    T1050_proc.add(u)
                    T1050_e1.append((u, v, attributes))
            except:
                pass

    T1050_events.append(T1050_e1)
#     print(len(T1050_proc))
#     print(len(T1050_e1))

In [ ]:
# T1044: File System Permissions Weakness
# Sysmon Event ID = 7 
# driver_loaded = "*\\Temp\\*" OR "C:\\Users\\*" OR 
# driver_signature_status != "Valid")
def searchT1044(T1044_proc, T1044_events):
    T1044_e1 = []
    dirs = ["\\Temp\\", "\\Users\\"]
    for edge in G.edges:
        u = edge[0]
        v = edge[1]
        attributes = G.edges[edge]['attributes']
        if G.nodes[v]["oType"] == "MODULE" and attributes['action'] == "LOAD":
            try:
                module_path = re.escape(attributes['module_path'])
                for key in dirs:
                    if key in module_path:
                        T1044_proc.add(u)
                        T1044_e1.append((u, v, attributes))
            except:
                pass

    T1044_events.append(T1044_e1)
#     print(len(T1044_proc))
#     print(len(T1044_e1))


In [ ]:
# T1138: Application Shimming
def searchT1138(T1138_proc, T1138_events):
    T1138_e1 = []
    T1138_e2 = []
    T1138_e3 = []
    # Event 1
    # Sysmon Event ID = 1
    # Windows Security Event ID = 4688
    # process_name = "sdbinst.exe"
    for edge in G.edges:
        u = edge[0]
        v = edge[1]
        attributes = G.edges[edge]['attributes']
        if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
            try:
                image_path = attributes['image_path']
                parent_path = attributes['parent_image_path']
                if "sdbinst.exe" in image_path:
                    T1138_proc.add(u)
                    T1138_e1.append((u, v, attributes))
            except:
                pass

    # Event 2
    # Sysmon Event ID = 11 
    # file_path = "C:\\Windows\\AppPatch\\Custom\\*"
    for edge in G.edges:
        u = edge[0]
        v = edge[1]
        attributes = G.edges[edge]['attributes']
        if G.nodes[v]["oType"] == "FILE":# and attributes['action'] == "OPEN":
            try:
                file_path = attributes['file_path']
                #parent_path = attributes['parent_image_path']
                if "C:\\Windows\\AppPatch\\Custom\\" in file_path:
                    T1138_proc.add(u)
                    T1138_e2.append((u, v, attributes))
            except:
                pass

    # Event 3
    # Sysmon Event ID = 12, 13, or 14
    # registry_key_path = "*\\SOFTWARE\\Microsoft\\Windows NT\\CurrentVersion\\AppCompatFlags\\InstalledSDB\\*"
    for edge in G.edges:
        u = edge[0]
        v = edge[1]
        attributes = G.edges[edge]['attributes']
        if G.nodes[v]["oType"] == "REGISTRY":# and attributes['action'] == "OPEN":
            try:
                registry_path = attributes['key']
                #parent_path = attributes['parent_image_path']
                if "\\SOFTWARE\\Microsoft\\Windows NT\\CurrentVersion\\AppCompatFlags\\InstalledSDB\\" in registry_path:
                    T1138_proc.add(u)
                    T1138_e3.append((u, v, attributes))
            except:
                pass

    T1138_events.append(T1138_e1)
    T1138_events.append(T1138_e2)
    T1138_events.append(T1138_e3)
#     print(len(T1138_proc))
#     print(len(T1138_e1))
#     print(len(T1138_e2))
#     print(len(T1138_e3))

In [ ]:
# T1103: AppInit DLLs
# Sysmon Event ID = 12, 13, or 14
# registry_key_path = "*\\SOFTWARE\\Microsoft\\Windows NT\\CurrentVersion\\Windows\\Appinit_Dlls\\*" OR
#                     "*\\SOFTWARE\\Wow6432Node\\Microsoft\\Windows NT\\CurrentVersion\\Windows\\Appinit_Dlls\\*"
def searchT1103(T1103_proc, T1103_events):
    T1103_e1 = []
    registry_keys = ["\\SOFTWARE\\Microsoft\\Windows NT\\CurrentVersion\\Windows\\Appinit_Dlls\\", 
                     "\\SOFTWARE\\Wow6432Node\\Microsoft\\Windows NT\\CurrentVersion\\Windows\\Appinit_Dlls\\"]
    for edge in G.edges:
        u = edge[0]
        v = edge[1]
        attributes = G.edges[edge]['attributes']
        if G.nodes[v]["oType"] == "REGISTRY":# and attributes['action'] == "OPEN":
            try:
                registry_path = re.escape(attributes['key'])
                #parent_path = attributes['parent_image_path']
                for key in registry_keys:
                    if key in registry_path:
                        T1103_proc.add(u)
                        T1103_e1.append((u, v, attributes))
            except:
                pass
    T1103_events.append(T1103_e1)
#     print(len(T1103_proc))
#     print(len(T1103_e1))

In [ ]:
# 1182 Appcert.dll
# registry_key_path="*\\System\\CurrentControlSet\\Control\\Session Manager\\AppCertDlls\\*"
def searchT1182(T1182_proc, T1182_events):
    registry_keys = ["\\System\\CurrentControlSet\\Control\\Session Manager\\AppCertDlls\\"]
    T1182_e1 = []
    for edge in G.edges:
        u = edge[0]
        v = edge[1]
        attributes = G.edges[edge]['attributes']
        if G.nodes[v]["oType"] == "REGISTRY":# and attributes['action'] == "OPEN":
            try:
                registry_path = re.escape(attributes['key'])
                #parent_path = attributes['parent_image_path']
                for key in registry_keys:
                    if key in registry_path:
                        T1182_proc.add(u)
                        T1182_e1.append((u, v, attributes))
            except:
                pass

    T1182_events.append(T1182_e1)
#     print(len(T1182_proc))
#     print(len(T1182_e1))

In [ ]:
# T1015: Accessibility Features 
def searchT1015(T1015_proc, T1015_events):
    T1015_e1 = []
    T1015_e2 = []
    # Event 1
    # Sysmon Event ID = 1
    # process_parent_name = "winlogon.exe"
    # process_name = ["sethc.exe", "utilman.exe", "osk.exe", "magnify.exe", "displayswitch.exe", "narrator.exe", "atbroker.exe"]
    image_dict = ["sethc.exe", "utilman.exe", "osk.exe", "magnify.exe", "displayswitch.exe", "narrator.exe", "atbroker.exe"]
    for edge in G.edges:
        u = edge[0]
        v = edge[1]
        attributes = G.edges[edge]['attributes']
        if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
            try:
                image_path = attributes['image_path']
                parent_path = attributes['parent_image_path']
                if "winlogon.exe" in parent_path:
                    for img in image_dict:
                        if img in image_path:
                            T1015_proc.add(u)
                            T1015_e1.append((u, v, attributes))
            except:
                pass


    # Event 2
    # Sysmon Event ID = 12, 13, or 14
    # registry_keys = ["HKLM\\SOFTWARE\\Microsoft\\Windows NT\\CurrentVersion\\Image File Execution Options\\"]
    registry_keys = ["HKLM\\SOFTWARE\\Microsoft\\Windows NT\\CurrentVersion\\Image File Execution Options\\"]
    for edge in G.edges:
        u = edge[0]
        v = edge[1]
        attributes = G.edges[edge]['attributes']
        if G.nodes[v]["oType"] == "REGISTRY":# and attributes['action'] == "OPEN":
            try:
                registry_path = re.escape(attributes['key'])
                #parent_path = attributes['parent_image_path']
                for key in registry_keys:
                    if key in registry_path:
                        T1015_proc.add(u)
                        T1015_e2.append((u, v, attributes))
            except:
                pass

    T1015_events.append(T1015_e1)
    T1015_events.append(T1015_e2)
#     print(len(T1015_proc))
#     print(len(T1015_e1))
#     print(len(T1015_e2))

In [ ]:
# Master script for privilege escalation - save em to file
# Not for now 

T1053_proc = set()
T1053_events = []
searchT1053(T1053_proc, T1053_events)

T1013_proc = set()
T1013_events = []
searchT1013(T1013_proc, T1013_events)

T1050_proc = set()
T1050_events = []
searchT1050(T1050_proc, T1050_events)

T1044_proc = set()
T1044_events = []
searchT1044(T1044_proc, T1044_events)

T1138_proc = set()
T1138_events = []
searchT1138(T1138_proc, T1138_events)

T1103_proc = set()
T1103_events = []
searchT1103(T1103_proc, T1103_events)

T1182_proc = set()
T1182_events = []
searchT1182(T1182_proc, T1182_events)

T1015_proc = set()
T1015_events = []
searchT1015(T1015_proc, T1015_events)

T1088_proc = set()
T1088_events = []
searchT1088(T1088_proc, T1088_events)

T1179_proc = set()
T1179_events = []
searchT1179(T1179_proc, T1179_events)

T1183_proc = set()
T1183_events = []
searchT1183(T1183_proc, T1183_events)

T1055_proc = set()
T1055_events = []
searchT1055(T1055_proc, T1055_events)


In [ ]:
print(len(T1053_proc))
print(len(T1053_events[0]))
print(len(T1053_events[1]))

# Execution 

In [ ]:
# T1059: Command-Line Interface
# Sysmon Event ID = 1
# Windows Security Event ID = 4688
# process_name = "cmd.exe"
T1059_proc = set()
T1059_e1 = []
for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
        try:
            image_path = attributes['image_path']
            parent_path = attributes['parent_image_path']
            if "cmd.exe" in image_path:
                T1059_proc.add(u)
                T1059_e1.append((u, v, attributes))
        except:
            pass

print(len(T1059_proc))
print(len(T1059_e1))

In [ ]:
# [T1086] PowerShell
T1086_proc = set()
T1086_e1 = []
T1086_e2 = []
# Event 1
# Sysmon Event ID = 1
# Windows Security Event ID = 4688
# process_command_line = "*.Download*" or "*Net.WebClient*"
for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
        try:
            command_line = attributes['command_line']
            if "Download" in command_line or "Net.WebClient" in command_line:
                T1086_proc.add(u)
                T1086_e1.append((u, v, attributes))
        except:
            pass

# Event 2
# Sysmon Event ID = 1
# Windows Security Event ID = 4688
# process_name = "powershell.exe" or "powershell_ise.exe" or "psexec.exe"
for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
        try:
            image_path = attributes['image_path']
            if "powershell.exe" in image_path or "powershell_ise.exe" in image_path or "psexec.exe" in image_path:
                T1086_proc.add(u)
                T1086_e2.append((u, v, attributes))
        except:
            pass


print(len(T1086_proc))
print(len(T1086_e1))
print(len(T1086_e2))

In [ ]:
# T1047: Windows Management Instrumentation
T1047_proc = set()
T1047_e1 = []
T1047_e2 = []
# Event 1
# Sysmon Event ID = 1
# Windows Security Event ID = 4688
# process_parent_path = "*\\wmiprvse.exe" OR 
# process_name = "wmic.exe" OR 
# process_command_line = "*wmic* "
for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
        try:
            parent_path = attributes['parent_image_path']
            image_path = attributes['image_path']
            command_line = attributes['command_line']
            if "wmiprvse.exe" in parent_path or "wmic.exe" in image_path or "wmic" in command_line:
                T1047_proc.add(u)
                T1047_e1.append((u, v, attributes))
        except:
            pass

# Event 2
# Sysmon Event ID = 3
# process_name = "wmic.exe" OR
# process_command_line = "*wmic* "

# Event 3
# Sysmon Event ID = 1
# Windows Security Event ID = 4688
# process_parent_path = "C:\\Windows\\System32\\svchost.exe" AND
# process_path = "C:\\WINDOWS\\system32\\wbem\\scrcons.exe"
for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
        try:
            parent_path = attributes['parent_image_path']
            image_path = attributes['image_path']
            command_line = attributes['command_line']
            if "svchost.exe" in parent_path and "scrcons.exe" in image_path:
                T1047_proc.add(u)
                T1047_e2.append((u, v, attributes))
        except:
            pass


print(len(T1047_proc))
print(len(T1047_e1))
print(len(T1047_e2))

In [ ]:
# T1002: Data Compressed 
T1002_proc = set()
T1002_e1 = []
T1002_e2 = []

# Event 1
# Sysmon Event ID = 1
# Windows Security Event ID = 4688
# process_name = "powershell.exe" AND process_command_line="*-Recurse | Compress-Archive*") OR 
# process_name = "rar.exe" AND process_command_line="rar*a*") OR
# process_name = "7z.exe" or "*zip.exe"
for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
        try:
            parent_path = attributes['parent_image_path']
            image_path = attributes['image_path']
            command_line = attributes['command_line']
            if "powershell.exe" in image_path and ("Recurse" in command_line or "Compress-Archive" in command_line):
                T1002_proc.add(u)
                T1002_e1.append((u, v, attributes))
            elif "rar.exe" in image_path and "rar" in command_line:
                T1002_proc.add(u)
                T1002_e1.append((u, v, attributes))
            elif "7z.exe" in pamge_path or "zip.exe" in image_path:
                T1002_proc.add(u)
                T1002_e1.append((u, v, attributes))
        except:
            pass

        
# Event 2
# Sysmon Event ID = 11
# file_name = ["*.zip", "*.rar", "*.arj", "*.gz", "*.tar", "*.tgz", "*.7z", "*.zip", "*.tar.gz", "*.bin"]
file_names = [".*\.zip", ".*\.rar", ".*\.arj", ".*\.gz", ".*\.tar", ".*\.tgz", ".*\.7z", ".*\.zip", ".*\.tar\.gz", ".*\.bin"]
for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    if G.nodes[v]["oType"] == "FILE":# and attributes['action'] == "OPEN":
        try:
            file_path = re.escape(attributes['file_path'])
            for ext in file_names:
                if bool(re.search(ext, file_path)):
                    T1002_proc.add(u)
                    T1002_e2.append((u, v, attributes))
        except:
            pass

print(len(T1002_proc))
print(len(T1002_e1))
print(len(T1002_e2))

# Initial Access 

In [ ]:
# T1093: Spearphishing Attachment 
T1093_proc = set()
T1093_e1 = []
T1093_e2 = []

# Event 1
# Sysmon Event ID = 11 
# file_name = ["*.docm", "*.xlsm", "*.pptm", "*.ps1", "*.py", "*.js", "*.vbs", "*.hta", "*.bat", "*.slk", "*.jspx", "*.cmd", "*.php", "*.pyw", "*.xla", "*.application", "*.potm", "*.csproj", "*.aspx", "*.exe"]
file_name = [".*\.docm", ".*\.xlsm", ".*\.pptm", ".*\.ps1", ".*\.py", ".*\.js", ".*\.vbs", ".*\.hta", ".*\.bat", ".*\.slk", ".*\.jspx", ".*\.cmd", ".*\.php", ".*\.pyw", ".*\.xla", ".*\.application", ".*\.potm", ".*\.csproj", ".*\.aspx", ".*\.exe"]
for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    if G.nodes[v]["oType"] == "FILE":# and attributes['action'] == "OPEN":
        try:
            file_path = re.escape(attributes['file_path'])
            for ext in file_names:
                if bool(re.search(ext, file_path)):
                    T1093_proc.add(u)
                    T1093_e1.append((u, v, attributes))
        except:
            pass

# Event 2
# Sysmon Event ID = 13 
# registry_key_path = "*trustrecords*", "*TargetObject=*Software\\Microsoft\\VBA\\7.1\\Common*"

registry_keys = ["trustrecords", "TargetObject=.*Software\\Microsoft\\VBA\\.*\\Common*"]
for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    if G.nodes[v]["oType"] == "REGISTRY":# and attributes['action'] == "OPEN":
        try:
            registry_path = re.escape(attributes['key'])
            for mykey in registry_keys:
                if bool(re.search(mykey, registry_path)):
                    T1093_proc.add(u)
                    T1093_e2.append((u, v, attributes))
        except:
            pass

print(len(T1093_proc))
print(len(T1093_e1))
print(len(T1093_e2))

In [ ]:
# T1131: Authentication package
# registry_key_path = "\\SYSTEM\\CurrentControlSet\\Control\\Lsa\\" AND 
# NOT (process_path = "C:\\WINDOWS\\system32\\lsass.exe", "C:\\Windows\\system32\\svchost.exe", "C:\\Windows\\system32\\services.exe")
T1131_proc = set()
T1131_e1 = []
registry_keys = ["\\SYSTEM\\CurrentControlSet\\Control\\Lsa\\"]
benign_paths = ["C:\\WINDOWS\\system32\\lsass.exe", "C:\\Windows\\system32\\svchost.exe", "C:\\Windows\\system32\\services.exe"]
for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    if G.nodes[v]["oType"] == "REGISTRY":# and attributes['action'] == "OPEN":
        try:
            registry_path = re.escape(attributes['key'])
            image_path = re.escape(attributes['image_path'])
            isCand = False
            for mykey in registry_keys:
                if mykey in registry_path:
                    isCand = True
            for img in benign_paths:
                if img in image_path:
                    isCand = False
            if isCand:
                T1131_proc.add(u)
                T1131_e1.append((u, v, attributes))
        except:
            pass

print(len(T1131_proc))
print(len(T1131_e1))

In [ ]:
# T1042: Change Default File Association
# Sysmon Event ID = 12, 13, or 14
# registry_key_path = ["*\\SOFTWARE\\Classes\\*\\*", "*\\SOFTWARE\\Microsoft\\Windows\\CurrentVersion\\Explorer\\GlobalAssocChangedCounter"]
T1042_proc = set()
T1042_e1 = []
registry_keys = ["\\SOFTWARE\\Classes\\", "\\SOFTWARE\\Microsoft\\Windows\\CurrentVersion\\Explorer\\GlobalAssocChangedCounter"]
for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    if G.nodes[v]["oType"] == "REGISTRY":# and attributes['action'] == "OPEN":
        try:
            registry_path = re.escape(attributes['key'])
            isCand = False
            for mykey in registry_keys:
                if mykey in registry_path:
                    isCand = True
            if isCand:
                T1042_proc.add(u)
                T1042_e1.append((u, v, attributes))
        except:
            pass

print(len(T1042_proc))
print(len(T1042_e1))

In [ ]:
# T1136: Create Account 
# process_command_line = ["*New-LocalUser*", "*net*user*add*"]
T1136_proc = set()
T1136_e1 = []
commands = ["New-LocalUser", "net.*user.*add"]
for edge in G.edges:
    u = edge[0]
    v = edge[1]
    attributes = G.edges[edge]['attributes']
    if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
        try:
            parent_path = attributes['parent_image_path']
            image_path = attributes['image_path']
            command_line = attributes['command_line']
            for cmd in commands:
                if re.search(bool(re.search(cmd, command_line))):
                    T1136_proc.add(u)
                    T1136_e1.append((u, v, attributes))
        except:
            pass
print(len(T1136_proc))
print(len(T1136_e1))

In [ ]:
# 